In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import warnings, os, gc, re
import matplotlib.pyplot as plt
import psutil
import swifter
# import icd10
from ast import literal_eval

# from plotnine import ggplot, aes, geom_line
from plotnine import *
# import pygal as pg

%matplotlib inline

warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.4f}'.format

In [2]:
# count number of rows
from itertools import (takewhile,repeat)
def rawincount(filename):
    f = open(filename, 'rb')
    bufgen = takewhile(lambda x: x, (f.raw.read(1024*1024) for _ in repeat(None)))
    return sum( buf.count(b'\n') for buf in bufgen )

In [3]:
path, dirs, files = next(os.walk("chunks/"))
directory = 'chunks/'
file_count = len(files)
year_start = 2013
year_end = 2013

In [4]:
# codes in ILLNESS*_CODE that are not ICD-10 codes
not_icd = ['MCP01', 'NSD01', 'P0000', 'P0001', 'ANC01', 'ANC02']

#icd-10 philhealth list - 2017
df_icd10 = pd.read_excel(os.path.join(directory, 'ICD10 philhealth.xlsx'))
df_icd10 = df_icd10.set_axis(['ILLNESS1_CODE','DESCRIPTION','GROUP','CASE_RATE','PROFESSIONAL_FEE', \
                              'HEALTH_CARE_INST_FEE'], axis=1, inplace=False)

#procedures philhealth list - 2015
df_procs = pd.read_excel(os.path.join(directory, 'Procedures philhealth.xlsx'))
df_procs = df_procs.set_axis(['ILLNESS1_CODE','DESCRIPTION','CASE_RATE','PROFESSIONAL_FEE', \
                              'HEALTH_CARE_INST_FEE'], axis=1, inplace=False)

In [61]:
# computes for ILLNESS*_CODE total counts
def illness_counts(dframe, \
                   not_in_icd, df_icd, df_proc, \
                   col_name_grp, \
                   tab_name_med, tab_name_proc, \
                   writer1):   
    df2 = dframe.loc[dframe[col_name_grp].astype(str).str[0].str.isalpha() & ~dframe[col_name_grp].isin(not_in_icd)] \
            .groupby([col_name_grp]).size().sort_values(ascending=False).to_frame().reset_index() \
            .set_axis([col_name_grp, 'Freq.'], axis=1, inplace=False)
    df2 = df2[~df2['Freq.'].isnull()]
    df2['Percent'] = ((df2['Freq.'].values/total_year)*100)
    df2['Freq.'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
#     df3 = df2[[~df2['Freq.'].isnull()]]
    df2.to_excel(writer1, sheet_name=tab_name_med, index=False)
    del[df2]
#     del[df3]

#     df2 = dframe.loc[dframe[col_name_grp].str.isnumeric() | dframe[col_name_grp].isin(not_in_icd)] \
    df2 = dframe.loc[~(dframe[col_name_grp].astype(str).str[0].str.isalpha() & ~dframe[col_name_grp].isin(not_in_icd))] \
            .groupby([col_name_grp]).size().sort_values(ascending=False).to_frame().reset_index() \
            .set_axis([col_name_grp, 'Freq.'], axis=1, inplace=False)
    df2 = df2[~df2['Freq.'].isnull()]    
    df2['Percent'] = ((df2['Freq.'].values/total_year)*100)
    df2['Freq.'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
#     df3 = df2[[~df2['Freq.'].isnull()]]
    df2.to_excel(writer1, sheet_name=tab_name_proc, index=False)
    del[df2]
#     del[df3]

# # computes for ILLNESS*_CODE AMOUNT totals 
# def illness_amts(dframe, \
#                  not_in_icd, df_icd, df_proc, \
#                  col_name_grp, amt_col, 
#                  tab_name_med, tab_name_proc, \
#                  tot_pd_amt, writer1): 
#     df2 = dframe.loc[dframe[col_name_grp].astype(str).str[0].str.isalpha() & ~dframe[col_name_grp].isin(not_in_icd)] \
#             .groupby([col_name_grp])[amt_col].sum().sort_values(ascending=False).to_frame().reset_index() \
#             .set_axis([col_name_grp, 'Total Amount of Claims'], axis=1, inplace=False)[~df2['Freq.'].isnull()] 
#     df2['Percent'] = ((df2['Total Amount of Claims'].values/tot_pd_amt)*100)
#     df2['Total Amount of Claims'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}' \
#                                         .format(x['Total Amount of Claims']),axis=1)
#     df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
#     df3 = df2.loc[[~df2['Freq.'].isnull()]]
#     df3.to_excel(writer1, sheet_name=tab_name_med, index=False)
#     del[df2]
#     del[df3]
        
#     df2 = dframe.loc[dframe[col_name_grp].str.isnumeric() | dframe[col_name_grp].isin(not_in_icd)] \
#             .groupby([col_name_grp])[amt_col].sum().sort_values(ascending=False).to_frame().reset_index() \
#             .set_axis([col_name_grp, 'Total Amount of Claims'], axis=1, inplace=False)[~df2['Freq.'].isnull()] 
#     df2['Percent'] = ((df2['Total Amount of Claims'].values/tot_pd_amt)*100)
#     df2['Total Amount of Claims'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}' \
#                                         .format(x['Total Amount of Claims']),axis=1)
#     df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
#     df3 = df2.loc[[~df2['Freq.'].isnull()]]
#     df3.to_excel(writer1, sheet_name=tab_name_proc, index=False)
#     del[df2]
#     del[df3]

# # computes for ILLNESS*_CODE totals, +1 col grouping besides illness code field
# def illness_counts_oth_1(dframe, 
#                 not_in_icd, df_icd, df_proc, \
#                 illness_cd_col, col_name_grp, \
#                 tab_name_med, tab_name_proc, \
#                 tot_year, writer1):
#     df2 = df1.loc[df1[illness_cd_col].astype(str).str[0].str.isalpha() & ~df1[illness_cd_col].isin(not_icd)] \
#             .groupby([col_name_grp, illness_cd_col]).size().to_frame().reset_index() \
#             .set_axis([col_name_grp, illness_cd_col, 'Freq.'], axis=1, inplace=False) \
#             .sort_values(by=[col_name_grp, 'Freq.'], ascending=[True, False])[~df2['Freq.'].isnull()] 
#     df2['Percent'] = ((df2['Freq.']/tot_year)*100)
#     df2['Freq.'] = df2.apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
#     df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)     
#     df3 = df2.loc[[~df2['Freq.'].isnull()]]
#     df3.to_excel(writer1, sheet_name=tab_name_med, index=False)
#     del[df2]
#     del[df3]

#     df2 = df1.loc[df1[illness_cd_col].str.isnumeric() | df1[illness_cd_col].isin(not_icd)] \
#             .groupby([col_name_grp, illness_cd_col]).size().to_frame().reset_index() \
#             .set_axis([col_name_grp,illness_cd_col, 'Freq.'], axis=1, inplace=False) \
#             .sort_values(by=[col_name_grp, 'Freq.'], ascending=[True, False])[~df2['Freq.'].isnull()] 
#     df2['Percent'] = ((df2['Freq.']/tot_year)*100)
#     df2['Freq.'] = df2.apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
#     df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
#     df3 = df2.loc[[~df2['Freq.'].isnull()]]
#     df3.to_excel(writer1, sheet_name=tab_name_proc, index=False)
#     del[df2]
#     del[df3]

# # computes for ILLNESS*_CODE amounts, +1 col grouping besides illness code field    
# def illness_amts_oth_1(dframe, 
#                 not_in_icd, df_icd, df_proc, \
#                 illness_cd_col, col_name_grp, amt_col, \
#                 tab_name_med, tab_name_proc, \
#                 tot_amt, writer1):    
#     df2 = df1.loc[df1[illness_cd_col].astype(str).str[0].str.isalpha() & ~df1[illness_cd_col].isin(not_icd)] \
#             .groupby([col_name_grp, illness_cd_col])[amt_col].sum().to_frame().reset_index() \
#             .set_axis([col_name_grp, illness_cd_col, 'Total Amount of Claims'], axis=1, inplace=False) \
#             .sort_values(by=[col_name_grp, 'Total Amount of Claims'], ascending=[True, False])[~df2['Freq.'].isnull()] 
#     df2['Percent'] = ((df2['Total Amount of Claims']/tot_amt)*100)
#     df2['Total Amount of Claims'] = df2.apply(lambda x: '{:,.2f}'.format(x['Total Amount of Claims']),axis=1)
#     df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
#     df3 = df2.loc[[~df2['Freq.'].isnull()]]
#     df3.to_excel(writer1, sheet_name=tab_name_med, index=False)
#     del[df2]
#     del[df3]

#     df2 = df1.loc[df1[illness_cd_col].str.isnumeric() | df1[illness_cd_col].isin(not_icd)] \
#             .groupby([col_name_grp, illness_cd_col])[amt_col].sum().to_frame().reset_index() \
#             .set_axis([col_name_grp, illness_cd_col, 'Total Amount of Claims'], axis=1, inplace=False) \
#             .sort_values(by=[col_name_grp, 'Total Amount of Claims'], ascending=[True, False])[~df2['Freq.'].isnull()] 
#     df2['Percent'] = ((df2['Total Amount of Claims']/tot_amt)*100)
#     df2['Total Amount of Claims'] = df2.apply(lambda x: '{:,.2f}'.format(x['Total Amount of Claims']),axis=1)
#     df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
#     df3 = df2.loc[[~df2['Freq.'].isnull()]]
#     df3.to_excel(writer1, sheet_name=tab_name_proc, index=False)
#     del[df2]
#     del[df3]

In [62]:
for i in range(year_start, year_end+1):
    df1 = pd.read_csv(os.path.join(directory, str(i) + '_new_1.csv'), low_memory=False, index_col=0)

    # for lighter pandas df and faster execution
    df1['PRO_NAME'] = df1['PRO_NAME'].astype('category')
    df1['INSTITUTION_NAME'] = df1['INSTITUTION_NAME'].astype('category')
    df1['OWNERSHIP'] = df1['OWNERSHIP'].astype('category')
    df1['INSTITUTION_CLASS'] = df1['INSTITUTION_CLASS'].astype('category')
    df1['INSTITUTION_PROVINCE'] = df1['INSTITUTION_PROVINCE'].astype('category')
    df1['INSTITUTION_MUNICIPALITY'] = df1['INSTITUTION_MUNICIPALITY'].astype('category')
    df1['MEM_CATEGORY'] = df1['MEM_CATEGORY'].astype('category')
    df1['MEM_SUB_CATEGORY'] = df1['MEM_SUB_CATEGORY'].astype('category')

    # computing for Turnaround Time
    df1[['CHECK_DT','RECEIVED_REFILED_DATE']] = df1[['CHECK_DT','RECEIVED_REFILED_DATE']].apply(pd.to_datetime)
    df1['TAT'] = (df1['CHECK_DT'] - df1['RECEIVED_REFILED_DATE']).dt.days

    # computing for Financial Coverage; set x/0 row as NaN so it will not be included in the aggregates
    df1['FINANCIAL_COVERAGE'] = df1['PAID_AMOUNT']/df1['ACTUAL_AMT']
    df1.loc[~np.isfinite(df1['FINANCIAL_COVERAGE']), 'FINANCIAL_COVERAGE'] = np.nan

    print(i)
    print(len(df1.index))
    total_year = len(df1.index)
    total_paid_amt = df1['PAID_AMOUNT'].sum()
    print('with illness code NaN:', len(df1[df1['ILLNESS1_CODE'].isnull()]))
    
    writer = pd.ExcelWriter(os.path.join(directory, str(i) + '_totals.xlsx'), engine='xlsxwriter')

    illness_counts(df1, not_icd, df_icd10, df_procs, 'ILLNESS1_CODE', \
                   'Med Claim - Counts', 'Proc Claim - Counts', writer)
#     illness_amts(df1, not_icd, df_icd10, df_procs, 'ILLNESS1_CODE', 'PAID_AMOUNT', \
#                  'Med Claim - Sum', 'Proc Claim - Sum', total_paid_amt, writer)
        

#     illness_counts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'INSTITUTION_CLASS', \
#                     'Med Claim - Counts-Inst Class', 'Proc Claim - Counts-Inst Class', \
#                     total_year, writer)    

#     illness_amts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'INSTITUTION_CLASS', 'PAID_AMOUNT', \
#                     'Med Claim - Amt-Inst Class', 'Proc Claim - Amt-Inst Class', \
#                     total_paid_amt, writer)

#     illness_counts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'PRO_NAME', \
#                     'Med Claim - Counts-Region', 'Proc Claim - Counts-Region', \
#                     total_year, writer)    

#     illness_amts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'PRO_NAME', 'PAID_AMOUNT', \
#                     'Med Claim - Amt-Region', 'Proc Claim - Amt-Region', \
#                     total_paid_amt, writer)
    
    writer.save()  
   
    del[df1]
    gc.collect()        

2013
5800283
with illness code NaN: 0
